# Taller resuelto: Selección de K en K-Means

Este notebook está resuelto con la estructura que hemos venido usando en clase:

1. **Antes de cada bloque de código** se explica qué se va a hacer.  
2. **Dentro del código** cada línea tiene comentarios.  
3. **Después del bloque** aparece la interpretación de lo observado.  

## Objetivo
Aplicar **K-Means** sobre datos sintéticos y un dataset real para aprender a seleccionar el número de clusters usando:
- **método del codo**
- **silhouette score**


## Bloque 1. Importar librerías

En este bloque vamos a cargar las librerías necesarias para:
- generar datos sintéticos,
- entrenar modelos K-Means,
- dividir datos en entrenamiento y prueba,
- y construir visualizaciones.


In [ ]:
# Importamos NumPy para trabajar con arreglos numéricos.
import numpy as np

# Importamos Matplotlib para construir gráficas.
import matplotlib.pyplot as plt

# Importamos KMeans desde scikit-learn.
from sklearn.cluster import KMeans

# Importamos make_blobs para generar datos sintéticos agrupados.
from sklearn.datasets import make_blobs

# Importamos train_test_split para separar datos.
from sklearn.model_selection import train_test_split


### Interpretación del bloque 1

Este bloque deja preparado el entorno de trabajo.

**Qué significa:**
- Ya tenemos listas las herramientas para generar datos, entrenar modelos de clustering y evaluar cómo cambia el agrupamiento al modificar `k`.
- En K-Means, elegir el número de clusters es una decisión crítica, por eso este notebook se enfoca en ese problema.


## Bloque 2. Generar datos isotrópicos y visualizarlos

En este bloque vamos a:
- crear un dataset sintético con grupos relativamente bien definidos,
- dividirlo en entrenamiento y prueba,
- y visualizar su estructura.


In [ ]:
# Generamos datos sintéticos con cuatro grupos.
x, y = make_blobs(
    n_samples=500,      # Número total de observaciones
    n_features=2,       # Dos variables para visualización en 2D
    centers=4,          # Número real de grupos
    cluster_std=1.2,    # Dispersión de cada grupo
    random_state=3      # Semilla para reproducibilidad
)

# Separamos los datos en entrenamiento y prueba.
X_train, X_test, y_train, y_test = train_test_split(
    x,                  # Variables de entrada
    y,                  # Etiquetas reales solo como referencia
    test_size=0.3,      # 30% para prueba
    random_state=42,    # Reproducibilidad
    stratify=y          # Mantener proporción de grupos
)

# Mostramos las dimensiones.
print("Shape de X_train:", X_train.shape)
print("Shape de X_test:", X_test.shape)

# Creamos una figura.
_, ax = plt.subplots(figsize=(7, 7))

# Dibujamos los datos con color según el grupo real.
ax.scatter(
    x[:, 0],            # Primera variable
    x[:, 1],            # Segunda variable
    c=y,                # Color según grupo
    cmap=plt.cm.Paired, # Paleta de colores
    edgecolors="k"      # Borde negro
)

# Agregamos título y ejes.
ax.set_title("Datos isotrópicos")
ax.set_xlabel("Variable 1")
ax.set_ylabel("Variable 2")

# Mostramos la figura.
plt.show()


### Interpretación del bloque 2

La gráfica muestra grupos relativamente compactos y separados.

**Qué debe observarse:**
- El dataset parece adecuado para K-Means porque los clusters son aproximadamente circulares o convexos.
- Aunque conocemos las etiquetas reales, el algoritmo no las usará para aprender.
- Este paso sirve para construir intuición visual antes de empezar a estimar `k`.


## Bloque 3. Crear una función de distancia euclidiana

En este bloque vamos a recordar la distancia que usa K-Means para asignar puntos al centroide más cercano.


In [ ]:
# Definimos una función para calcular la distancia euclidiana entre dos puntos 2D.
def euclidean_distance(pt1, pt2):
    # Verificamos que el primer punto tenga dos coordenadas.
    assert len(pt1) == 2, "Error! No 2d point"

    # Verificamos que el segundo punto tenga dos coordenadas.
    assert len(pt2) == 2, "Error! No 2d point"

    # Calculamos la distancia euclidiana.
    return ((pt1[0] - pt2[0])**2 + (pt1[1] - pt2[1])**2) ** 0.5

# Probamos la función con un ejemplo conocido.
print("Distancia entre [0,0] y [3,4]:", euclidean_distance([0, 0], [3, 4]))


### Interpretación del bloque 3

Este bloque refuerza la idea geométrica del algoritmo.

**Qué significa:**
- K-Means asigna cada punto al centroide más cercano.
- Esa cercanía normalmente se mide con distancia euclidiana.
- Recordar esta distancia ayuda a entender por qué el algoritmo genera ciertas fronteras entre clusters.


## Bloque 4. Aplicar el método del codo

En este bloque vamos a:
- probar varios valores de `k`,
- entrenar un modelo K-Means para cada uno,
- medir la inercia,
- y visualizar la curva del codo.


In [ ]:
# Definimos los valores de k que vamos a evaluar.
clusters = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Creamos una lista para guardar las inercias.
elbow_distances = []

# Recorremos cada valor de k.
for k in clusters:
    # Creamos el modelo K-Means.
    kmeans = KMeans(
        n_clusters=k,       # Número de clusters
        init="k-means++",   # Inicialización inteligente
        n_init=10,          # Número de inicializaciones
        random_state=42     # Reproducibilidad
    )

    # Entrenamos el modelo con datos de entrenamiento.
    kmeans.fit(X_train)

    # Guardamos la inercia.
    elbow_distances.append(kmeans.inertia_)

# Creamos una figura.
_, ax = plt.subplots(figsize=(7, 7))

# Dibujamos la curva del codo.
ax.plot(clusters, elbow_distances, marker="x")

# Agregamos título y ejes.
ax.set_title("Elbow")
ax.set_xlabel("Número de clusters (k)")
ax.set_ylabel("Inercia")

# Mostramos la gráfica.
plt.show()


### Interpretación del bloque 4

La curva del codo ayuda a identificar un valor razonable de `k`.

**Qué debe buscarse:**
- Un punto donde la reducción de inercia deja de ser tan fuerte.
- Ese “quiebre” indica que agregar más clusters ya no mejora significativamente el agrupamiento.

**Mensaje docente:**
La inercia siempre baja al aumentar `k`, pero el objetivo no es minimizarla sin control, sino encontrar equilibrio entre simplicidad y ajuste.


## Bloque 5. Aplicar el método silhouette

En este bloque vamos a calcular el silhouette score para distintos valores de `k` y comparar la calidad de los clusters.


In [ ]:
# Importamos silhouette_score.
from sklearn.metrics import silhouette_score

# Definimos los valores de k a evaluar.
clusters = [2, 3, 4, 5, 6, 7, 8, 9, 10]

# Creamos una lista para guardar los resultados.
silhouette_distances = []

# Recorremos cada valor de k.
for k in clusters:
    # Creamos el modelo K-Means.
    kmeans = KMeans(
        n_clusters=k,       # Número de clusters
        init="k-means++",   # Inicialización inteligente
        n_init=10,          # Número de inicializaciones
        random_state=42     # Reproducibilidad
    )

    # Entrenamos el modelo.
    kmeans.fit(X_train)

    # Calculamos silhouette score.
    score = silhouette_score(X_train, kmeans.labels_)

    # Guardamos el resultado.
    silhouette_distances.append(score)

# Creamos una figura.
_, ax = plt.subplots(figsize=(7, 7))

# Dibujamos la curva de silhouette.
ax.plot(clusters, silhouette_distances, marker="x")

# Agregamos título y ejes.
ax.set_title("Silhouette")
ax.set_xlabel("Número de clusters (k)")
ax.set_ylabel("Silhouette score")

# Mostramos la gráfica.
plt.show()


### Interpretación del bloque 5

El silhouette score mide la calidad del agrupamiento considerando:
- **cohesión** interna del cluster
- **separación** respecto a otros clusters

**Cómo se interpreta:**
- valores cercanos a 1: clusters bien definidos
- valores cercanos a 0: clusters confusos
- valores negativos: mala asignación

**Idea clave:**
Mientras el codo se enfoca en compactación, silhouette agrega una mirada más completa sobre la separación.


## Bloque 6. Cargar un dataset real: Iris

En este bloque vamos a repetir el análisis sobre un dataset real clásico del Machine Learning.


In [ ]:
# Importamos el dataset Iris.
from sklearn.datasets import load_iris

# Cargamos el dataset.
data = load_iris()

# Guardamos las variables en x.
x = data.data

# Guardamos las clases reales como referencia.
y = data.target

# Dibujamos dos variables del dataset.
plt.figure(figsize=(7, 5))
plt.scatter(
    x[:, 0],             # Primera variable
    x[:, 1],             # Segunda variable
    c=y,                 # Color según clase real
    cmap=plt.cm.Paired   # Paleta
)

# Mostramos dimensiones.
print(x.shape, y.shape)

# Mostramos la gráfica.
plt.show()


### Interpretación del bloque 6

Este bloque cambia de un escenario sintético a uno real.

**Qué significa esto:**
- Ya no trabajamos con grupos generados artificialmente.
- Ahora el reto es descubrir si K-Means puede encontrar una estructura similar a las clases conocidas.
- Este paso es importante porque acerca el análisis a casos más realistas.


## Bloque 7. Crear funciones para elbow y silhouette

En este bloque vamos a encapsular el cálculo del codo y silhouette en funciones reutilizables.


In [ ]:
# Definimos una función para calcular inercia para varios valores de k.
def elbow_kmeans(clusters, data):
    # Creamos una lista vacía.
    elbow_distances = []

    # Recorremos cada valor de k.
    for k in clusters:
        # Creamos el modelo K-Means.
        model = KMeans(
            n_clusters=k,
            init="k-means++",
            n_init=10,
            random_state=42
        )

        # Entrenamos el modelo.
        model.fit(data)

        # Guardamos la inercia.
        elbow_distances.append(model.inertia_)

    # Retornamos la lista.
    return elbow_distances

# Definimos una función para calcular silhouette score para varios valores de k.
def silhouette_kmeans(clusters, data):
    # Creamos una lista vacía.
    silhouette_distances = []

    # Recorremos cada valor de k.
    for k in clusters:
        # Creamos el modelo K-Means.
        model = KMeans(
            n_clusters=k,
            init="k-means++",
            n_init=10,
            random_state=42
        )

        # Entrenamos el modelo.
        model.fit(data)

        # Calculamos silhouette score.
        score = silhouette_score(data, model.labels_)

        # Guardamos el score.
        silhouette_distances.append(score)

    # Retornamos la lista.
    return silhouette_distances


### Interpretación del bloque 7

Este bloque hace el notebook más profesional.

**Qué aporta:**
- Evita repetir código.
- Permite reutilizar el análisis en cualquier dataset.
- Acerca el trabajo a una práctica real de analítica y desarrollo.


## Bloque 8. Comparar elbow y silhouette en Iris

En este bloque vamos a visualizar juntos ambos criterios para tomar una decisión más sólida sobre `k`.


In [ ]:
# Definimos los valores de k para elbow.
clusters_elbow = [1, 2, 3, 4, 5, 6, 7, 8]

# Calculamos la curva del codo.
elbow = elbow_kmeans(clusters_elbow, x)

# Definimos los valores de k para silhouette.
clusters_silhouette = [2, 3, 4, 5, 6, 7, 8]

# Calculamos los scores.
silhouette = silhouette_kmeans(clusters_silhouette, x)

# Creamos una figura con dos subgráficas.
_, axes = plt.subplots(1, 2, figsize=(13, 5))

# Dibujamos elbow.
axes[0].plot(clusters_elbow, elbow, marker="x")
axes[0].set_title("Elbow")

# Dibujamos silhouette.
axes[1].plot(clusters_silhouette, silhouette, marker="x")
axes[1].set_title("Silhouette")

# Mostramos la figura.
plt.show()


### Interpretación del bloque 8

Esta comparación ayuda a tomar una decisión más fundamentada.

**Qué debe analizarse:**
- Si ambos métodos apuntan a un valor parecido, eso fortalece la elección.
- Si difieren, el analista debe justificar cuál criterio pesa más según el problema.
- En Iris, el valor óptimo suele acercarse a 3, pero lo importante es aprender a justificar la decisión.


## Bloque 9. Entrenar el modelo final y visualizar clusters

En este bloque vamos a elegir un valor final de `k`, entrenar el modelo y visualizar el agrupamiento.


In [ ]:
# Elegimos el valor final de k.
k_final = 3

# Entrenamos el modelo final.
kmeans = KMeans(
    n_clusters=k_final,
    init="k-means++",
    n_init=10,
    random_state=42
).fit(x)

# Guardamos los centroides.
centroids = kmeans.cluster_centers_

# Mostramos los centroides.
print(centroids)

# Predecimos el cluster de cada observación.
predictions = kmeans.predict(x)

# Creamos la figura.
_, ax = plt.subplots(figsize=(7, 5))

# Dibujamos los puntos según cluster encontrado.
ax.scatter(
    x[:, 0],
    x[:, 1],
    c=predictions,
    cmap="viridis",
    edgecolors="k"
)

# Dibujamos los centroides.
ax.scatter(
    centroids[:, 0],
    centroids[:, 1],
    c="red",
    s=250,
    marker="X",
    label="Centroides"
)

# Agregamos título y leyenda.
ax.set_title("Clusters encontrados en Iris")
ax.legend()

# Mostramos la figura.
plt.show()


### Interpretación del bloque 9

La gráfica final muestra el resultado del clustering sobre Iris.

**Qué debe observarse:**
- Los clusters encontrados pueden parecerse a las clases reales, pero no necesariamente coinciden perfectamente.
- Eso es normal, porque clustering no trabaja con etiquetas reales.
- El objetivo aquí no es reproducir exactamente las clases, sino descubrir estructura útil.


## Cierre del ejercicio

### Ideas clave
- K-Means necesita que el analista elija `k`.
- El método del codo ayuda a ver compactación.
- El silhouette score ayuda a ver separación.
- No existe una regla única: la selección de `k` es una decisión analítica.
